# Notebook 1 — Segment images with Cellpose 4.0.6 and save masks

## Installation 

### for Mac
Latest stable release is Cellpose 4.0.6, [available via conda-forge](https://anaconda.org/conda-forge/cellpose)

```bash
conda env create -f ./envs/cellpose.yml
conda activate cellpose
```

### for colab

`%pip install "cellpose==4.0.6" "torch" "torchvision" "torchaudio" "scikit-image>=0.22.0" "tqdm>=4.66.0" "pandas>=2.2.0"`

In [1]:
# Cell 1 — Imports and Apple Silicon device
from pathlib import Path
from typing import List
import os
import numpy as np
from skimage import io, exposure
from tqdm import tqdm
import torch
from cellpose import models

def has_mps() -> bool:
    return hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

device = torch.device("mps") if has_mps() else torch.device("cpu")
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # safe fallback for rare ops
print("torch:", torch.__version__)
print("device:", device)




Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	darwin 
python version: 	3.10.18 
torch version:  	2.7.1! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


torch: 2.7.1
device: mps


In [2]:
# Cell 2 — Paths and parameters
project_root = Path("/Users/ashi/github/cm4ai_codefest2025")

# Inputs: data/<channel>/*.jpg  (filenames end with *_<channel>.jpg)
img_root = project_root / "data"
channels: List[str] = ["yellow"]     # segment yellow only

# Outputs
masks_root = project_root / "analysis" / "cellpose_results2"

# Image extensions to search
image_exts = [".jpg", ".jpeg", ".tif", ".tiff", ".png"]

# Model
pretrained_model = "cpsam"

# Segmentation params
use_auto_diameter = True      # True => diameter=None (auto)
diameter = None if use_auto_diameter else 40.0

# Very permissive start
cellprob_threshold = -10.0
flow_threshold = 0.0

invert = False
batch_size = 4

# Preprocessing
do_rescale_intensity = True
do_clahe = False
clahe_clip = 2.0
clahe_tiles = (8, 8)

print("- project root:", project_root)
print("- image root:", img_root)
print("- channels:", channels)
print("- masks root:", masks_root)
print("- model:", pretrained_model)
print("- diameter:", diameter, "(auto)" if use_auto_diameter else "")
print("- thresholds: cellprob", cellprob_threshold, "| flow", flow_threshold)
print("- preprocess: rescale_intensity", do_rescale_intensity, "| clahe", do_clahe)


- project root: /Users/ashi/github/cm4ai_codefest2025
- image root: /Users/ashi/github/cm4ai_codefest2025/data
- channels: ['yellow']
- masks root: /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2
- model: cpsam
- diameter: None (auto)
- thresholds: cellprob -10.0 | flow 0.0
- preprocess: rescale_intensity True | clahe False


In [3]:
# Cell 3 — Utilities
def discover_images(folder: Path, exts) -> list[Path]:
    files = []
    for ext in exts:
        files.extend(sorted(folder.glob(f"*{ext}")))
    # de-dup by stem
    seen, uniq = set(), []
    for p in files:
        if p.stem not in seen:
            uniq.append(p); seen.add(p.stem)
    return uniq

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

def preprocess_rgb(img: np.ndarray) -> np.ndarray:
    """Per-image intensity rescue for faint signals."""
    if img.ndim == 2:  # grayscale
        arr = img
        if do_rescale_intensity:
            arr = exposure.rescale_intensity(arr)
        if do_clahe:
            arrf = exposure.equalize_adapthist(arr, clip_limit=clahe_clip, nbins=256, kernel_size=clahe_tiles)
            arr = (arrf * np.iinfo(img.dtype).max).astype(img.dtype)
        return arr
    # RGB path
    out = img.copy()
    if do_rescale_intensity:
        # rescale each channel independently
        for c in range(out.shape[-1]):
            out[..., c] = exposure.rescale_intensity(out[..., c])
    if do_clahe:
        # CLAHE per channel in float [0,1]
        for c in range(out.shape[-1]):
            outf = exposure.equalize_adapthist(out[..., c], clip_limit=clahe_clip, nbins=256, kernel_size=clahe_tiles)
            out[..., c] = (outf * np.iinfo(img.dtype).max).astype(img.dtype)
    return out


In [4]:
# Cell 4 — Load Cellpose 4.x model
model = models.CellposeModel(
    gpu=False,
    pretrained_model=pretrained_model,
    device=device
)
print("Loaded model:", model.pretrained_model)


Loaded model: /Users/ashi/.cellpose/models/cpsam


In [ ]:
# Cell 5 — First run with your original settings (flow_threshold = 0.0)
total_images = 0
failed = []

for ch in channels:  # only "yellow"
    in_dir = img_root / ch
    out_dir = masks_root / ch / "png"
    ensure_dir(out_dir)

    if not in_dir.exists():
        print(f"[SKIP] Missing channel folder: {in_dir}")
        continue

    files = discover_images(in_dir, image_exts)
    if not files:
        print(f"[WARN] No images in {in_dir} with {image_exts}")
        continue

    print(f"[{ch}] {len(files)} images -> {out_dir}")

    for group in tqdm(list(chunked(files, batch_size)), desc=f"Seg {ch}"):
        # read + preprocess
        raw_imgs = [io.imread(p) for p in group]
        imgs = [preprocess_rgb(im) for im in raw_imgs]

        try:
            masks, flows, styles = model.eval(
                imgs,
                diameter=diameter,            # None => auto
                batch_size=len(imgs),
                channel_axis=-1,              # RGB
                invert=invert,
                normalize=False,              # disable tile normalization
                rescale=True,                 # keep exactly as your original
                flow_threshold=flow_threshold,
                cellprob_threshold=cellprob_threshold,
            )
        except Exception as e:
            for p in group:
                failed.append((p.name, str(e)))
            continue

        for p, m in zip(group, masks):
            out_path = out_dir / f"{p.stem}_masks.png"   # keeps *_yellow in stem
            io.imsave(out_path, m.astype(np.uint16), check_contrast=False)

    total_images += len(files)

print(f"Done. Segmented {total_images} yellow images.")
print(f"Masks written under: {masks_root}/yellow/png/")
if failed:
    print("Failures:", failed[:5], "... total:", len(failed))


In [ ]:
# Cell 6 — Rerun with only one change: stronger flow_threshold to suppress tiny specks
# This single change encourages smoother, more contiguous masks and reduces fragment noise.
flow_threshold = 0.1
print("flow_threshold set to", flow_threshold)

# Repeat the same segmentation loop, unchanged except for flow_threshold
total_images = 0
failed = []

for ch in channels:  # only "yellow"
    in_dir = img_root / ch
    out_dir = masks_root / ch / "png"
    ensure_dir(out_dir)

    files = discover_images(in_dir, image_exts)
    print(f"[{ch}] {len(files)} images -> {out_dir}")

    for group in tqdm(list(chunked(files, batch_size)), desc=f"Seg {ch} [flow=0.3]"):
        raw_imgs = [io.imread(p) for p in group]
        imgs = [preprocess_rgb(im) for im in raw_imgs]

        try:
            masks, flows, styles = model.eval(
                imgs,
                diameter=diameter,
                batch_size=len(imgs),
                channel_axis=-1,
                invert=invert,
                normalize=False,
                rescale=True,                 # unchanged
                flow_threshold=flow_threshold, # only change
                cellprob_threshold=cellprob_threshold,
            )
        except Exception as e:
            for p in group:
                failed.append((p.name, str(e)))
            continue

        for p, m in zip(group, masks):
            out_path = out_dir / f"{p.stem}_masks.png"
            io.imsave(out_path, m.astype(np.uint16), check_contrast=False)

    total_images += len(files)

print(f"Done. Segmented {total_images} yellow images with flow_threshold=0.3.")
print(f"Masks written under: {masks_root}/yellow/png/")
if failed:
    print("Failures:", failed[:5], "... total:", len(failed))


In [ ]:
# Cell 6B — Rerun with only one change: invert = True
invert = True
print("invert set to", invert)

total_images = 0
failed = []

for ch in channels:
    in_dir = img_root / ch
    out_dir = masks_root / ch / "png"
    ensure_dir(out_dir)

    files = discover_images(in_dir, image_exts)
    print(f"[{ch}] {len(files)} images -> {out_dir}")

    for group in tqdm(list(chunked(files, batch_size)), desc=f"Seg {ch} [invert=True]"):
        raw_imgs = [io.imread(p) for p in group]
        imgs = [preprocess_rgb(im) for im in raw_imgs]

        try:
            masks, flows, styles = model.eval(
                imgs,
                diameter=diameter,
                batch_size=len(imgs),
                channel_axis=-1,
                invert=invert,                # only change here
                normalize=False,
                rescale=True,
                flow_threshold=flow_threshold,
                cellprob_threshold=cellprob_threshold,
            )
        except Exception as e:
            for p in group:
                failed.append((p.name, str(e)))
            continue

        for p, m in zip(group, masks):
            io.imsave((masks_root / ch / "png" / f"{p.stem}_masks.png"),
                      m.astype(np.uint16), check_contrast=False)

    total_images += len(files)

print(f"Done. Segmented {total_images} yellow images with invert=True.")
if failed:
    print("Failures:", failed[:5], "total:", len(failed))


In [ ]:
# Cell 6D — Make masks much larger with a big fixed diameter
use_auto_diameter = False
diameter = 80.0   # try 160 first; if still small -> 200; if too big -> 120
print("use_auto_diameter =", use_auto_diameter, " | diameter =", diameter)


total_images = 0
failed = []

for ch in channels:
    in_dir = img_root / ch
    out_dir = masks_root / ch / "png"
    ensure_dir(out_dir)

    files = discover_images(in_dir, image_exts)
    print(f"[{ch}] {len(files)} images -> {out_dir}")

    for group in tqdm(list(chunked(files, batch_size)), desc=f"Seg {ch} [fixed diam=40]"):
        raw_imgs = [io.imread(p) for p in group]
        imgs = [preprocess_rgb(im) for im in raw_imgs]

        try:
            masks, flows, styles = model.eval(
                imgs,
                diameter=diameter,            # only effective change
                batch_size=len(imgs),
                channel_axis=-1,
                invert=invert,
                normalize=False,
                rescale=True,
                flow_threshold=flow_threshold,
                cellprob_threshold=cellprob_threshold,
            )
        except Exception as e:
            for p in group:
                failed.append((p.name, str(e)))
            continue

        for p, m in zip(group, masks):
            io.imsave((masks_root / ch / "png" / f"{p.stem}_masks.png"),
                      m.astype(np.uint16), check_contrast=False)

    total_images += len(files)

print(f"Done. Segmented {total_images} yellow images with fixed diameter.")
if failed:
    print("Failures:", failed[:5], "total:", len(failed))


In [10]:
# Keep what worked:
use_auto_diameter = False
diameter = 80.0
flow_threshold = 0.0
invert = False

# ONE CHANGE:
normalize_imgs = True  # <-- turn on Cellpose normalization
print("normalize =", normalize_imgs)


total_images = 0
failed = []

for ch in channels:
    in_dir = img_root / ch
    out_dir = masks_root / ch / "png"
    ensure_dir(out_dir)

    files = discover_images(in_dir, image_exts)
    print(f"[{ch}] {len(files)} images -> {out_dir}")

    for group in tqdm(list(chunked(files, batch_size)), desc=f"Seg {ch} [fixed diam=40]"):
        raw_imgs = [io.imread(p) for p in group]
        imgs = [preprocess_rgb(im) for im in raw_imgs]

        try:
            masks, flows, styles = model.eval(
                imgs,
                diameter=diameter,            # only effective change
                batch_size=len(imgs),
                channel_axis=-1,
                invert=invert,
                normalize=normalize_imgs,
                rescale=True,
                flow_threshold=flow_threshold,
                cellprob_threshold=cellprob_threshold,
            )
        except Exception as e:
            for p in group:
                failed.append((p.name, str(e)))
            continue

        for p, m in zip(group, masks):
            io.imsave((masks_root / ch / "png" / f"{p.stem}_masks.png"),
                      m.astype(np.uint16), check_contrast=False)

    total_images += len(files)

print(f"Done. Segmented {total_images} yellow images with fixed diameter.")
if failed:
    print("Failures:", failed[:5], "total:", len(failed))


normalize = True
[yellow] 10 images -> /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/yellow/png


Seg yellow [fixed diam=40]:   0%|          | 0/3 [00:00<?, ?it/s]rescaling deprecated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Seg yellow [fixed diam=40]:  33%|███▎      | 1/3 [00:17<00:35, 17.85s/it]rescaling deprecated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Seg yellow [fixed diam=40]:  67%|██████▋   | 2/3 [00:35<00:17, 17.68s/it]rescaling deprecated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Seg yellow [fixed diam=40]: 100%|██████████| 3/3 [00:44<00:00, 14.86s/it]

Done. Segmented 10 yellow images with fixed diameter.


In [11]:
from skimage import color

for group in tqdm(list(chunked(files, batch_size)), desc=f"Seg {ch}"):
    raw_imgs = [io.imread(p) for p in group]
    imgs = [preprocess_rgb(im) for im in raw_imgs]

    masks, flows, styles = model.eval(
        imgs,
        diameter=diameter,
        batch_size=len(imgs),
        channel_axis=-1,
        invert=invert,
        normalize=normalize_imgs,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
    )

    # Always loop across every file in the batch
    for p, m, im in zip(group, masks, raw_imgs):
        out_mask = out_dir / f"{p.stem}_masks.png"
        io.imsave(out_mask, m.astype(np.uint16), check_contrast=False)

        # --- binary preview (guaranteed non-empty) ---
        bin_prev = (m > 0).astype(np.uint8) * 255
        io.imsave(out_dir / f"{p.stem}_mask_preview.png", bin_prev, check_contrast=False)

        # --- overlay preview (always saves) ---
        base = preprocess_rgb(im)
        basef = base.astype(np.float32)
        if basef.max() > 0:
            basef /= basef.max()
        overlay = color.label2rgb(m, image=basef, bg_label=0, alpha=0.35)
        io.imsave(out_dir / f"{p.stem}_overlay.png", (overlay*255).astype(np.uint8), check_contrast=False)


Seg yellow:   0%|          | 0/3 [00:00<?, ?it/s]Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Seg yellow:  33%|███▎      | 1/3 [00:22<00:44, 22.48s/it]Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Seg yellow:  67%|██████▋   | 2/3 [00:44<00:22, 22.26s/it]Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Seg yellow: 100%|██████████| 3/3 [00:56<00:00, 18.67s/it]


In [14]:
# Cell — merged segmentation + previews (v4-clean)
from skimage import color
import numpy as np

# Keep what worked
use_auto_diameter = False
diameter = 60
flow_threshold = 0.0
invert = False

# One change you wanted: turn on Cellpose normalization
normalize_imgs = True
print(f"normalize = {normalize_imgs} | diameter = {diameter} | flow = {flow_threshold} | invert = {invert}")

total_images = 0
failed = []

for ch in channels:
    in_dir = img_root / ch
    out_dir = masks_root / ch / "png"
    ensure_dir(out_dir)

    files = discover_images(in_dir, image_exts)
    if not files:
        print(f"[WARN] No images in {in_dir} with {image_exts}")
        continue

    print(f"[{ch}] {len(files)} images -> {out_dir}")

    for group in tqdm(list(chunked(files, batch_size)), desc=f"Seg {ch} [diam={diameter}, norm={normalize_imgs}]"):
        raw_imgs = [io.imread(p) for p in group]
        imgs = [preprocess_rgb(im) for im in raw_imgs]

        try:
            # v4 API: no 'rescale' kwarg
            masks, flows, styles = model.eval(
                imgs,
                diameter=diameter,
                batch_size=len(imgs),
                channel_axis=-1,          # RGB inputs
                invert=invert,
                normalize=normalize_imgs, # True enables CP v4 tile normalization
                flow_threshold=flow_threshold,
                cellprob_threshold=cellprob_threshold,  # keep your current value from earlier cell
            )
        except Exception as e:
            # Still write blank previews so every file has outputs
            for p, im in zip(group, raw_imgs):
                io.imsave(out_dir / f"{p.stem}_masks.png",
                          np.zeros(im.shape[:2], np.uint16), check_contrast=False)
                # binary preview (empty)
                io.imsave(out_dir / f"{p.stem}_mask_preview.png",
                          np.zeros(im.shape[:2], np.uint8), check_contrast=False)
                # overlay (just normalized image)
                base = preprocess_rgb(im).astype(np.float32)
                if base.max() > 0: base /= base.max()
                io.imsave(out_dir / f"{p.stem}_overlay.png",
                          (base*255).astype(np.uint8), check_contrast=False)
            failed.append((";".join([p.name for p in group]), str(e)))
            continue

        # Always loop across the whole batch and save all three outputs
        for p, m, im in zip(group, masks, raw_imgs):
            # mask (uint16 labels; looks dark in grayscale viewers — normal)
            io.imsave(out_dir / f"{p.stem}_masks.png",
                      m.astype(np.uint16), check_contrast=False)

            # binary preview (white where any label > 0)
            bin_prev = (m > 0).astype(np.uint8) * 255
            io.imsave(out_dir / f"{p.stem}_mask_preview.png",
                      bin_prev, check_contrast=False)

            # overlay preview
            base = preprocess_rgb(im).astype(np.float32)
            if base.max() > 0: base /= base.max()
            overlay = color.label2rgb(m, image=base, bg_label=0, alpha=0.35)
            io.imsave(out_dir / f"{p.stem}_overlay.png",
                      (overlay*255).astype(np.uint8), check_contrast=False)

        total_images += len(group)

print(f"Done. Segmented {total_images} images with diameter={diameter}, normalize={normalize_imgs}.")
if failed:
    print("Failures:", failed[:2], "... total:", len(failed))
print("Outputs in:", masks_root / "yellow" / "png")

normalize = True | diameter = 60 | flow = 0.0 | invert = False
[yellow] 10 images -> /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/yellow/png


Seg yellow [diam=60, norm=True]:   0%|          | 0/3 [00:00<?, ?it/s]Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Seg yellow [diam=60, norm=True]:  33%|███▎      | 1/3 [00:27<00:55, 27.90s/it]Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Seg yellow [diam=60, norm=True]:  67%|██████▋   | 2/3 [00:54<00:27, 27.27s/it]Resizing is depricated in v4.0.1+
Resizing is depricated in v4.0.1+
Seg yellow [diam=60, norm=True]: 100%|██████████| 3/3 [01:08<00:00, 22.87s/it]

Done. Segmented 10 images with diameter=60, normalize=True.
Outputs in: /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/yellow/png
